# Crawl tiếp corpus Member A trên Kaggle (VI/ZH)

Notebook này chạy trực tiếp trên Kaggle, không cần repository local. Bật **Internet** trong Notebook options và thêm Kaggle Dataset chứa `links_corpus.parquet`, `query.parquet` (tùy chọn) cùng checkpoint trước đó (nếu resume). Cell chuẩn bị sẽ dò file trong `/kaggle/input`.

Notebook chỉ crawl 44 hostname Member A được liệt kê trong cell cấu hình. Bảng tổng hợp hiển thị **Tổng URL / Đã xong / Còn lại** và các trạng thái `blocked`, `too_short`, `ok`, `empty`, `http_error`... Tiến độ trong lúc chạy được cập nhật trên một dòng; nhãn giải thích bằng tiếng Việt, mã trạng thái giữ nguyên để khớp log.

Mỗi URL được commit vào SQLite sau khi xử lý. Khi dừng cell bình thường, notebook xuất kết quả và tạo `/kaggle/working/member_a_resume.zip`, gồm checkpoint cùng tài liệu/log. **Để resume ở phiên Kaggle hôm sau:** chờ cell kết thúc sau khi bấm Stop/Interrupt, dùng **Save Version** để lưu Output, rồi thêm Output của phiên trước làm Input cho notebook mới. Cell chuẩn bị sẽ tự tìm và khôi phục bundle. Kaggle không tự giữ `/kaggle/working` sau khi phiên bị đóng; nếu phiên bị kill đột ngột, hãy lưu/đính kèm output gần nhất trước khi đóng phiên.

Cần kiểm tra quyền truy cập và điều khoản website. Crawler đọc `robots.txt`, tôn trọng crawl-delay và tốc độ tối thiểu 1 giây giữa request cùng hostname; không vượt CAPTCHA, paywall hoặc block. Hostname có lỗi robots/HTTP chặn sẽ được tạm dừng, URL chưa thử của host vẫn ở trạng thái chờ.

In [ ]:
%pip -q install aiohttp protego trafilatura langdetect zstandard pyarrow

In [ ]:
from __future__ import annotations

import asyncio
import hashlib
import io
import json
import os
import re
import shutil
import sqlite3
import time
import unicodedata
import zipfile
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import quote, urljoin, urlsplit
from urllib.request import Request, urlopen
from typing import Any

import aiohttp
import pyarrow as pa
import pyarrow.parquet as pq
import trafilatura
import zstandard
from langdetect import DetectorFactory, detect_langs
from langdetect.lang_detect_exception import LangDetectException
from protego import Protego

DetectorFactory.seed = 0

# Đầu vào: thêm corpus và checkpoint trước vào Kaggle Input.
INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working/member_a_crawl")
DOCS_DIR = WORK_ROOT / "docs" / "a"
LOGS_DIR = WORK_ROOT / "crawl_log" / "a"
STATE_PATH = WORK_ROOT / "state-a-current.sqlite3"
RESUME_BUNDLE = Path("/kaggle/working/member_a_resume.zip")
for directory in (WORK_ROOT, DOCS_DIR, LOGS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

# Tìm parquet theo cấu trúc Kaggle Dataset hoặc tên file.
link_candidates = sorted(INPUT_ROOT.rglob("links_corpus.parquet"))
if not link_candidates:
    raise FileNotFoundError("Không tìm thấy links_corpus.parquet trong /kaggle/input.")
preferred_links = [p for p in link_candidates if "corpus" in p.parts and "raw" in p.parts]
INPUT_PATH = preferred_links[0] if preferred_links else link_candidates[0]
query_candidates = sorted(INPUT_ROOT.rglob("query.parquet"))
QUERY_PATH = query_candidates[0] if query_candidates else None

# Khôi phục bundle Output đã đính kèm từ phiên Kaggle trước.
if not STATE_PATH.exists():
    bundles = sorted(INPUT_ROOT.rglob("member_a_resume.zip"), key=lambda p: p.stat().st_mtime, reverse=True)
    if bundles:
        with zipfile.ZipFile(bundles[0]) as archive:
            for member in archive.infolist():
                name = Path(member.filename).name
                safe_path = Path(member.filename)
                if ".." in safe_path.parts or safe_path.is_absolute():
                    continue
                if name == "state-a-current.sqlite3" or member.filename.startswith(("docs/a/", "crawl_log/a/")):
                    target = WORK_ROOT / safe_path
                    target.parent.mkdir(parents=True, exist_ok=True)
                    with archive.open(member) as source, target.open("wb") as output:
                        shutil.copyfileobj(source, output)
        print(f"Đã khôi phục checkpoint từ: {bundles[0]}")

MEMBER_A_DOMAINS = set("""
cnkang.com ask.39.net zysjonline.com zhongyibaodian.net nhathuoclongchau.com.vn
zydcd.com youlai.cn laodong.vn phunusuckhoe.giadinhonline.vn bingli.iiyi.com
qihuangzhishu.com baohaiphong.vn vietnamnet.vn test.pmphai.com tiemchunglongchau.com.vn
baocantho.com.vn woman.39.net baoangiang.com.vn vov.vn khoahocphothong.vn youmed.vn
nk.39.net msdmanuals.cn article.iiyi.com baoquangtri.vn man.39.net baovinhlong.com.vn
tamanhhospital.vn tnb.39.net tuoitre.vn qdnd.vn baogialai.com.vn ek.39.net
baoquangninh.vn baidu.com baochinhphu.vn vnexpress.net sggp.org.vn pf.39.net
health.people.com.cn hanoimoi.vn baotayninh.vn byby.39.net benhvienvietduc.org
""".split())
EXCLUDED_DOMAINS = {
    "baohaiphong.vn", "baotayninh.vn", "khoahocphothong.vn", "laodong.vn",
    "nhathuoclongchau.com.vn", "youmed.vn", "vov.vn",
}
CONFIRMED_ACCESS_DOMAINS: set[str] = set()
GLOBAL_CONCURRENCY = 16
REQUESTS_PER_DOMAIN = 1.0
MIN_TEXT_CHARS = 200
MAX_RESPONSE_BYTES = 5_000_000
MAX_RETRIES = 3
USER_AGENT = "R2AI2026CorpusResearch/1.0 (respectful research crawler)"
SCHEMA_VERSION = "1.0"
NORM_VERSION = "n1"
SHARD_SIZE = 20_000
PROGRESS_SECONDS = 5

links_file = pq.ParquetFile(INPUT_PATH)
if not {"id", "url"}.issubset(links_file.schema_arrow.names):
    raise ValueError("links_corpus.parquet phải có hai cột id và url.")
query_count = 0
if QUERY_PATH is not None:
    query_file = pq.ParquetFile(QUERY_PATH)
    if not {"id", "query"}.issubset(query_file.schema_arrow.names):
        raise ValueError("query.parquet phải có hai cột id và query.")
    query_count = query_file.metadata.num_rows

print(f"Đầu vào: {links_file.metadata.num_rows:,} URL | câu truy vấn: {query_count:,} dòng")
print(f"Tệp links: {INPUT_PATH}")
if QUERY_PATH is None:
    print("Không tìm thấy query.parquet; tệp này chỉ dùng để kiểm tra, không cần cho crawl.")
print(f"Hostname Member A: {len(MEMBER_A_DOMAINS)} | đồng thời tối đa: {GLOBAL_CONCURRENCY}")

CONTROL = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f-\x9f\u00ad\u180e\u200b-\u200f\u202a-\u202e\u2060-\u206f\ufeff]")
SPACES = re.compile(r"[^\S\n]+")
NEWLINES = re.compile(r"\n{3,}")
LOG_SCHEMA = pa.schema([
    ("uid", pa.string()), ("id", pa.int64()), ("url", pa.string()),
    ("domain", pa.string()), ("owner", pa.string()), ("status", pa.string()),
    ("http_status", pa.int64()), ("attempt", pa.int64()),
    ("fetched_at", pa.string()), ("bytes", pa.int64()),
    ("final_url", pa.string()), ("error", pa.string()), ("shard", pa.string()),
])


def normalize_text(value: str) -> str:
    value = unicodedata.normalize("NFC", value).replace("\r\n", "\n").replace("\r", "\n")
    return NEWLINES.sub("\n\n", SPACES.sub(" ", CONTROL.sub("", value))).strip()


def normalize_hostname(url: str) -> str | None:
    try:
        host = urlsplit(url).hostname
    except (TypeError, ValueError):
        return None
    if not host:
        return None
    host = host.lower().rstrip(".")
    return host[4:] if host.startswith("www.") else host


def utc_now() -> str:
    return datetime.now(timezone.utc).astimezone().isoformat(timespec="seconds")


# SQLite lưu sau từng URL để giữ kết quả đã hoàn tất khi dừng cell.
db = sqlite3.connect(STATE_PATH, timeout=60)
db.row_factory = sqlite3.Row
db.execute("PRAGMA journal_mode=WAL")
db.execute("PRAGMA synchronous=FULL")
db.executescript("""
CREATE TABLE IF NOT EXISTS urls(
    id INTEGER PRIMARY KEY, url TEXT NOT NULL, domain TEXT NOT NULL,
    status TEXT NOT NULL DEFAULT 'pending', attempt INTEGER NOT NULL DEFAULT 0,
    log_json TEXT, doc_json TEXT, log_exported INTEGER NOT NULL DEFAULT 0,
    doc_exported INTEGER NOT NULL DEFAULT 0, doc_shard TEXT
);
CREATE INDEX IF NOT EXISTS urls_pending ON urls(status,domain,id);
CREATE TABLE IF NOT EXISTS hashes(content_hash TEXT PRIMARY KEY,uid TEXT NOT NULL);
CREATE TABLE IF NOT EXISTS domain_blocks(domain TEXT PRIMARY KEY,reason TEXT NOT NULL);
CREATE TABLE IF NOT EXISTS metadata(key TEXT PRIMARY KEY,value TEXT NOT NULL);
""")
if "doc_shard" not in {r[1] for r in db.execute("PRAGMA table_info(urls)")}:
    db.execute("ALTER TABLE urls ADD COLUMN doc_shard TEXT")
    db.commit()

# Tránh dùng nhầm checkpoint với snapshot khác.
source_stat = INPUT_PATH.stat()
fingerprint = f"{source_stat.st_size}:{source_stat.st_mtime_ns}"
old_fingerprint = db.execute("SELECT value FROM metadata WHERE key='source_fingerprint'").fetchone()
if old_fingerprint and old_fingerprint["value"] != fingerprint:
    raise ValueError("Snapshot links đã đổi so với checkpoint. Hãy dùng bundle/checkpoint đúng snapshot; không ghi đè state cũ.")
db.execute("INSERT OR IGNORE INTO metadata(key,value) VALUES('source_fingerprint',?)", (fingerprint,))
db.commit()

# Lập chỉ mục 44 hostname; URL đã có từ bundle giữ nguyên trạng thái.
scanned = 0
assigned = 0
inserted = 0
for batch in links_file.iter_batches(columns=["id", "url"], batch_size=50_000):
    rows = []
    for row in batch.to_pylist():
        scanned += 1
        host = normalize_hostname(row["url"])
        if host in MEMBER_A_DOMAINS:
            assigned += 1
            rows.append((int(row["id"]), row["url"], host))
    if rows:
        before = db.total_changes
        db.executemany("INSERT OR IGNORE INTO urls(id,url,domain) VALUES(?,?,?)", rows)
        db.commit()
        inserted += db.total_changes - before
print(f"Đã lập chỉ mục {assigned:,}/{scanned:,} URL Member A; URL mới: {inserted:,}")

# Có thể nhập checkpoint SQLite cũ nếu được đính kèm làm Kaggle Input.
# Chỉ chuyển kết quả khi ID và URL cùng khớp snapshot hiện tại.
if db.execute("SELECT value FROM metadata WHERE key='old_state_migrated'").fetchone() is None:
    old_states = [p for p in INPUT_ROOT.rglob("state-a.sqlite3") if p.resolve() != STATE_PATH.resolve()]
    for old_path in old_states:
        legacy = sqlite3.connect(f"file:{old_path}?mode=ro", uri=True)
        try:
            tables = {r[0] for r in legacy.execute("SELECT name FROM sqlite_master WHERE type='table'")}
            if "urls" not in tables:
                continue
            columns = {r[1] for r in legacy.execute("PRAGMA table_info(urls)")}
            if not {"id", "url", "status", "log_json"}.issubset(columns):
                continue
            has_doc_json = "doc_json" in columns
            doc_select = "doc_json" if has_doc_json else "NULL"
            rows = legacy.execute(f"SELECT id,url,status,attempt,log_json,{doc_select} FROM urls WHERE status='done'").fetchall()
            with db:
                db.executemany("""
                    UPDATE urls SET status='done',attempt=?,log_json=?,doc_json=?
                    WHERE id=? AND url=? AND status='pending'
                """, [(r[3],r[4],r[5],r[0],r[1]) for r in rows])
            if "content_hashes" in tables:
                hash_columns = {r[1] for r in legacy.execute("PRAGMA table_info(content_hashes)")}
                if {"content_hash", "uid"}.issubset(hash_columns):
                    hashes = legacy.execute("SELECT content_hash,uid FROM content_hashes").fetchall()
                    with db:
                        db.executemany("INSERT OR IGNORE INTO hashes VALUES(?,?)", hashes)
            print(f"Đã đối chiếu checkpoint SQLite cũ: {old_path}")
        finally:
            legacy.close()
    db.execute("INSERT INTO metadata(key,value) VALUES('old_state_migrated','1')")
    db.commit()


# Đồng bộ lịch sử document Member A đã công khai trên Hugging Face.
HF_REPO = "ai-guru-medical-retrieval/multilingual-medical-retrieval-corpus"
HF_DOCS_API = f"https://huggingface.co/api/datasets/{HF_REPO}/tree/main/corpus/viz/docs/a?recursive=true&expand=false"
HF_DOCS_BASE = f"https://huggingface.co/datasets/{HF_REPO}/resolve/main/"
if db.execute("SELECT value FROM metadata WHERE key='hf_docs_imported'").fetchone() is None:
    request = Request(HF_DOCS_API, headers={"User-Agent": USER_AGENT})
    with urlopen(request, timeout=60) as response:
        hf_files = json.load(response)
    shard_paths = sorted(
        item["path"] for item in hf_files
        if item.get("type") == "file"
        and item.get("path", "").startswith("corpus/viz/docs/a/")
        and item["path"].endswith(".jsonl.zst")
    )
    if not shard_paths:
        raise RuntimeError("Không tìm thấy shard document Member A trên Hugging Face.")
    db.execute("DROP TABLE IF EXISTS temp.hf_docs")
    db.execute("CREATE TEMP TABLE hf_docs(id INTEGER,url TEXT,status TEXT,content_hash TEXT,uid TEXT,fetched_at TEXT,http_status INTEGER,PRIMARY KEY(id,url)) WITHOUT ROWID")
    hf_doc_count = 0
    for shard_path in shard_paths:
        shard_url = HF_DOCS_BASE + quote(shard_path, safe="/") + "?download=true"
        shard_tmp = WORK_ROOT / ("hf-" + Path(shard_path).name + ".download")
        request = Request(shard_url, headers={"User-Agent": USER_AGENT})
        with urlopen(request, timeout=120) as response, shard_tmp.open("wb") as output:
            shutil.copyfileobj(response, output)
        try:
            with shard_tmp.open("rb") as compressed, zstandard.ZstdDecompressor().stream_reader(compressed) as reader:
                text_reader = io.TextIOWrapper(reader, encoding="utf-8")
                batch = []
                for line in text_reader:
                    doc = json.loads(line)
                    if doc.get("owner") != "a" or doc.get("source") != "btc_url":
                        continue
                    try:
                        source_id = int(doc["doc_id"])
                    except (KeyError, TypeError, ValueError):
                        continue
                    source_url = doc.get("url")
                    fetch = doc.get("fetch") or {}
                    status = fetch.get("status")
                    if not isinstance(source_url, str) or status not in {"ok", "duplicate"}:
                        continue
                    content_hash = str(doc.get("content_hash") or "")
                    if content_hash.startswith("sha1:"):
                        content_hash = content_hash[5:]
                    batch.append((source_id, source_url, status, content_hash or None, str(doc.get("uid") or ""), fetch.get("fetched_at"), fetch.get("http_status")))
                    if len(batch) >= 2_000:
                        db.executemany("INSERT OR REPLACE INTO hf_docs VALUES(?,?,?,?,?,?,?)", batch)
                        hf_doc_count += len(batch)
                        batch.clear()
                if batch:
                    db.executemany("INSERT OR REPLACE INTO hf_docs VALUES(?,?,?,?,?,?,?)", batch)
                    hf_doc_count += len(batch)
            print(f"Đã đọc lịch sử Hugging Face: {Path(shard_path).name} | tổng document hợp lệ {hf_doc_count:,}")
        finally:
            shard_tmp.unlink(missing_ok=True)
    with db:
        matched_hf_docs = db.execute("""
            UPDATE urls SET status='done',attempt=0,doc_exported=1,log_exported=1,
                log_json=(SELECT json_object('uid',h.uid,'id',h.id,'url',h.url,'domain',urls.domain,'owner','a','status',h.status,'http_status',h.http_status,'attempt',0,'fetched_at',h.fetched_at,'bytes',0,'final_url',h.url,'error',NULL,'shard','huggingface') FROM hf_docs h WHERE h.id=urls.id AND h.url=urls.url)
            WHERE status='pending' AND EXISTS (SELECT 1 FROM hf_docs h WHERE h.id=urls.id AND h.url=urls.url)
        """).rowcount
        db.execute("""
            INSERT OR IGNORE INTO hashes(content_hash,uid)
            SELECT h.content_hash,h.uid FROM hf_docs h JOIN urls u ON u.id=h.id AND u.url=h.url
            WHERE u.status='done' AND h.content_hash IS NOT NULL AND h.uid!=''
        """)
        db.execute("INSERT INTO metadata(key,value) VALUES('hf_docs_imported',?)", (json.dumps({"shards": len(shard_paths), "documents": hf_doc_count}),))
    print(f"Đã đối chiếu {matched_hf_docs:,} URL đã crawl từ Hugging Face (khớp cả ID và URL).")
    if matched_hf_docs < hf_doc_count:
        print(f"Lưu ý: {hf_doc_count - matched_hf_docs:,} document không khớp chính xác snapshot links hiện tại nên chưa được đánh dấu hoàn tất.")


def domain_summary() -> list[sqlite3.Row]:
    return db.execute("""
        SELECT domain,COUNT(*) AS total,
          SUM(status='done') AS done,
          SUM(status!='done') AS remaining,
          SUM(json_extract(log_json,'$.status')='blocked') AS blocked,
          SUM(json_extract(log_json,'$.status')='too_short') AS too_short,
          SUM(json_extract(log_json,'$.status')='ok') AS ok,
          SUM(json_extract(log_json,'$.status')='empty') AS empty,
          SUM(json_extract(log_json,'$.status')='http_error') AS http_error,
          SUM(json_extract(log_json,'$.status')='timeout') AS timeout,
          SUM(json_extract(log_json,'$.status')='lang_mismatch') AS lang_mismatch,
          SUM(json_extract(log_json,'$.status')='duplicate') AS duplicate
        FROM urls GROUP BY domain ORDER BY remaining DESC,total DESC,domain
    """).fetchall()


def show_summary() -> None:
    rows = domain_summary()
    columns = [
        ("total", "Tổng URL"), ("done", "Đã xong"), ("remaining", "Còn lại"),
        ("blocked", "Bị chặn"), ("too_short", "Nội dung ngắn"), ("ok", "Đạt"),
        ("empty", "Rỗng"), ("http_error", "Lỗi HTTP"), ("timeout", "Quá giờ"),
        ("lang_mismatch", "Sai ngôn ngữ"), ("duplicate", "Trùng lặp"),
    ]
    print("Miền | " + " | ".join(label for _,label in columns))
    for row in rows:
        print(row["domain"] + " | " + " | ".join(f"{int(row[key] or 0):,}" for key,_ in columns))
    total = sum(int(row["total"]) for row in rows)
    done = sum(int(row["done"] or 0) for row in rows)
    print(f"TỔNG MEMBER A | {total:,} URL | đã xong {done:,} | còn lại {total-done:,}")

show_summary()

robots_cache: dict[str, Protego | None] = {}
robots_denied: set[str] = set()
domain_locks: defaultdict[str, asyncio.Lock] = defaultdict(asyncio.Lock)
domain_last_request: dict[str, float] = {}
request_semaphore = asyncio.Semaphore(GLOBAL_CONCURRENCY)


async def _wait_domain(domain: str, delay: float) -> None:
    remaining = delay - (time.monotonic() - domain_last_request.get(domain, 0.0))
    if remaining > 0:
        await asyncio.sleep(remaining)


async def _load_robots(session: aiohttp.ClientSession, domain: str, url: str) -> Protego | None:
    if domain in robots_cache:
        return robots_cache[domain]
    origin = urlsplit(url)
    robots_url = f"{origin.scheme}://{origin.netloc}/robots.txt"
    await _wait_domain(domain, REQUESTS_PER_DOMAIN)
    try:
        async with request_semaphore:
            async with session.get(robots_url, allow_redirects=False) as response:
                domain_last_request[domain] = time.monotonic()
                if response.status in {404, 410}:
                    robots_cache[domain] = None
                    return None
                if response.status != 200:
                    robots_denied.add(domain)
                    return None
                content = await response.text(errors="replace")
        policy = Protego.parse(content)
        robots_cache[domain] = policy
        return policy
    except (aiohttp.ClientError, asyncio.TimeoutError, ValueError):
        robots_denied.add(domain)
        return None


async def fetch_page(session: aiohttp.ClientSession, url: str, domain: str) -> dict[str, Any]:
    async with domain_locks[domain]:
        policy = await _load_robots(session, domain, url)
        if domain in robots_denied:
            return {"status":"blocked","attempt":0,"http_status":None,"fetched_at":utc_now(),"nbytes":0,"final_url":url,"error":"robots.txt unavailable or rejected; fail-closed","host_blocked":True,"body":None,"encoding":None}
        if policy is not None and not policy.can_fetch(url, user_agent=USER_AGENT):
            return {"status":"blocked","attempt":0,"http_status":None,"fetched_at":utc_now(),"nbytes":0,"final_url":url,"error":"disallowed by robots.txt","host_blocked":False,"body":None,"encoding":None}
        raw_crawl_delay = policy.crawl_delay(user_agent=USER_AGENT) if policy is not None else None
        crawl_delay = float(raw_crawl_delay) if isinstance(raw_crawl_delay, (int, float)) else 0.0
        delay = max(REQUESTS_PER_DOMAIN, crawl_delay)
        last_error = None
        last_status = None
        max_attempts = MAX_RETRIES + 1
        for attempt in range(1, max_attempts + 1):
            current_url = url
            retry = False
            redirects = 0
            while True:
                await _wait_domain(domain, delay)
                try:
                    async with request_semaphore:
                        async with session.get(current_url, allow_redirects=False) as response:
                            domain_last_request[domain] = time.monotonic()
                            fetched_at = utc_now()
                            if response.status in {301,302,303,307,308}:
                                location = response.headers.get("Location")
                                if not location or redirects >= 5:
                                    return {"status":"http_error","attempt":attempt,"http_status":response.status,"fetched_at":fetched_at,"nbytes":0,"final_url":str(response.url),"error":"redirect missing Location or exceeded 5 hops","host_blocked":False,"body":None,"encoding":response.charset}
                                target = urljoin(str(response.url), location)
                                if normalize_hostname(target) != domain:
                                    return {"status":"blocked","attempt":attempt,"http_status":response.status,"fetched_at":fetched_at,"nbytes":0,"final_url":target,"error":"cross-domain redirect not followed","host_blocked":False,"body":None,"encoding":response.charset}
                                if policy is not None and not policy.can_fetch(target, user_agent=USER_AGENT):
                                    return {"status":"blocked","attempt":attempt,"http_status":response.status,"fetched_at":fetched_at,"nbytes":0,"final_url":target,"error":"redirect disallowed by robots.txt","host_blocked":False,"body":None,"encoding":response.charset}
                                current_url = target
                                redirects += 1
                                continue
                            body = await response.content.read(MAX_RESPONSE_BYTES + 1)
                            last_status = response.status
                            if len(body) > MAX_RESPONSE_BYTES:
                                return {"status":"http_error","attempt":attempt,"http_status":response.status,"fetched_at":fetched_at,"nbytes":len(body),"final_url":str(response.url),"error":"response exceeds MAX_RESPONSE_BYTES","host_blocked":False,"body":None,"encoding":response.charset}
                            if 200 <= response.status < 300:
                                challenge = body[:8192].lower()
                                if b"captcha" in challenge or b"verify you are human" in challenge:
                                    return {"status":"blocked","attempt":attempt,"http_status":response.status,"fetched_at":fetched_at,"nbytes":len(body),"final_url":str(response.url),"error":"possible captcha/access challenge","host_blocked":True,"body":None,"encoding":response.charset}
                                return {"status":"ok","attempt":attempt,"http_status":response.status,"fetched_at":fetched_at,"nbytes":len(body),"final_url":str(response.url),"error":None,"host_blocked":False,"body":body,"encoding":response.charset or "utf-8"}
                            if response.status == 403:
                                return {"status":"blocked","attempt":attempt,"http_status":response.status,"fetched_at":fetched_at,"nbytes":len(body),"final_url":str(response.url),"error":"HTTP 403; access not bypassed","host_blocked":True,"body":None,"encoding":response.charset}
                            if response.status in {408,429} or response.status >= 500:
                                last_error = f"HTTP {response.status}"
                                if attempt < max_attempts:
                                    await asyncio.sleep(2 ** attempt)
                                    retry = True
                                    break
                            return {"status":"http_error","attempt":attempt,"http_status":response.status,"fetched_at":fetched_at,"nbytes":len(body),"final_url":str(response.url),"error":f"HTTP {response.status}","host_blocked":False,"body":None,"encoding":response.charset}
                except asyncio.CancelledError:
                    raise
                except (aiohttp.ClientError, asyncio.TimeoutError) as exc:
                    domain_last_request[domain] = time.monotonic()
                    last_error = f"{type(exc).__name__}: {exc}"
                    if attempt < max_attempts:
                        await asyncio.sleep(2 ** attempt)
                        retry = True
                        break
                    status = "timeout" if isinstance(exc, asyncio.TimeoutError) else "http_error"
                    return {"status":status,"attempt":attempt,"http_status":None,"fetched_at":utc_now(),"nbytes":0,"final_url":current_url,"error":last_error,"host_blocked":False,"body":None,"encoding":None}
            if retry:
                continue
        status = "timeout" if last_error and "TimeoutError" in last_error else "http_error"
        return {"status":status,"attempt":max_attempts,"http_status":last_status,"fetched_at":utc_now(),"nbytes":0,"final_url":url,"error":last_error or "request failed","host_blocked":False,"body":None,"encoding":None}


def make_records(source_id: int, url: str, domain: str, result: dict[str, Any]) -> tuple[dict[str, Any], dict[str, Any] | None]:
    status = str(result["status"])
    doc: dict[str, Any] | None = None
    if status == "ok" and result["body"] is not None:
        raw = bytes(result["body"]).decode(str(result["encoding"] or "utf-8"), errors="replace")
        article = normalize_text(trafilatura.extract(raw, include_comments=False, include_tables=True, include_links=False, output_format="txt") or "")
        metadata = trafilatura.extract_metadata(raw)
        title = normalize_text((metadata.title or "") if metadata else "")
        content = normalize_text(f"{title}\n\n{article}" if title else article)
        if not article:
            status = "empty"
        elif len(content) < MIN_TEXT_CHARS:
            status = "too_short"
        else:
            try:
                language_guess = detect_langs(content[:5000])
            except LangDetectException:
                language_guess = []
            detected = language_guess[0] if language_guess else None
            lang = "zh" if detected and detected.lang.startswith("zh") else detected.lang if detected else ""
            if lang not in {"vi", "zh"}:
                status = "lang_mismatch"
            else:
                digest = hashlib.sha1(content.encode("utf-8")).hexdigest()
                uid = f"{lang}:{source_id}"
                passages: list[dict[str, Any]] = []
                body_start = content.find(article)
                if title and content.startswith(title):
                    passages.append({"type":"title","start":0,"end":len(title)})
                if body_start >= 0 and article:
                    passages.append({"type":"body","start":body_start,"end":body_start+len(article)})
                doc = {"schema_version":SCHEMA_VERSION,"uid":uid,"doc_id":str(source_id),"lang":lang,"source":"btc_url","url":str(result["final_url"] or url),"domain":domain,"title":title,"text":content,"norm_version":NORM_VERSION,"passages":passages,"text_len":len(content),"content_hash":f"sha1:{digest}","dup_of":None,"fetch":{"status":status,"http_status":result["http_status"],"fetched_at":result["fetched_at"],"extractor":f"trafilatura@{trafilatura.__version__}","lang_detected":lang,"lang_conf":float(detected.prob) if detected else 0.0},"pubmed":None,"owner":"a","code_commit":None}
    log: dict[str, Any] = {"uid":str(doc["uid"]) if doc else f"src:{source_id}","id":source_id,"url":url,"domain":domain,"owner":"a","status":status,"http_status":result["http_status"],"attempt":result["attempt"],"fetched_at":result["fetched_at"],"bytes":result["nbytes"],"final_url":result["final_url"],"error":result["error"],"shard":None}
    if doc is not None:
        doc["fetch"]["status"] = status
    return log, doc


def next_sequence(directory: Path, prefix: str) -> int:
    numbers = []
    for path in directory.glob(f"{prefix}-*.*"):
        try:
            numbers.append(int(path.name.split("-")[-1].split(".")[0]))
        except ValueError:
            continue
    return max(numbers,default=0)+1


def export_ready(force: bool=False) -> None:
    doc_rows = db.execute("SELECT id,doc_json FROM urls WHERE status='done' AND doc_json IS NOT NULL AND doc_exported=0 ORDER BY id LIMIT ?",(SHARD_SIZE,)).fetchall()
    log_rows = db.execute("SELECT id,log_json,doc_shard FROM urls WHERE status='done' AND log_exported=0 ORDER BY id LIMIT ?",(SHARD_SIZE,)).fetchall()
    if force or len(doc_rows) >= SHARD_SIZE:
        if doc_rows:
            seq = next_sequence(DOCS_DIR,"viz-a")
            filename = f"viz-a-{seq:05d}.jsonl.zst"
            path, temp = DOCS_DIR/filename, DOCS_DIR/(filename+".tmp")
            with temp.open("wb") as raw:
                with zstandard.ZstdCompressor(level=3).stream_writer(raw,closefd=False) as out:
                    for row in doc_rows:
                        doc = json.loads(row["doc_json"])
                        out.write((json.dumps(doc,ensure_ascii=False)+"\n").encode("utf-8"))
                raw.flush(); os.fsync(raw.fileno())
            os.replace(temp,path)
            db.executemany("UPDATE urls SET doc_exported=1,doc_shard=? WHERE id=?",[(filename,int(r["id"])) for r in doc_rows]); db.commit()
    if force or len(log_rows) >= SHARD_SIZE:
        if log_rows:
            seq = next_sequence(LOGS_DIR,"log-a")
            filename = f"log-a-{seq:05d}.parquet"
            records = []
            for row in log_rows:
                record = json.loads(row["log_json"])
                record["shard"] = row["doc_shard"]
                records.append(record)
            temp = LOGS_DIR/(filename+".tmp")
            pq.write_table(pa.Table.from_pylist(records,schema=LOG_SCHEMA),temp,compression="zstd")
            os.replace(temp,LOGS_DIR/filename)
            db.executemany("UPDATE urls SET log_exported=1 WHERE id=?",[(int(r["id"]),) for r in log_rows]); db.commit()


async def crawl_all() -> None:
    timeout = aiohttp.ClientTimeout(total=25)
    connector = aiohttp.TCPConnector(limit=GLOBAL_CONCURRENCY)
    headers = {"User-Agent":USER_AGENT}
    stop_hosts = {r["domain"] for r in db.execute("SELECT domain FROM domain_blocks")}
    stop_hosts -= CONFIRMED_ACCESS_DOMAINS
    session_counts: Counter[str] = Counter()
    started = time.monotonic()
    last_report = started
    async with aiohttp.ClientSession(connector=connector,timeout=timeout,headers=headers) as session:
        async def crawl_domain(host: str) -> None:
            nonlocal last_report
            rows = db.execute("SELECT id,url FROM urls WHERE domain=? AND status='pending' ORDER BY id",(host,)).fetchall()
            for row in rows:
                if host in stop_hosts or host in EXCLUDED_DOMAINS:
                    return
                try:
                    result = await fetch_page(session,str(row["url"]),host)
                    log,doc = await asyncio.to_thread(make_records,int(row["id"]),str(row["url"]),host,result)
                    with db:
                        if doc is not None:
                            existing_hash = db.execute("SELECT uid FROM hashes WHERE content_hash=?", (doc["content_hash"],)).fetchone()
                            if existing_hash:
                                doc["dup_of"] = existing_hash["uid"]
                                doc["fetch"]["status"] = "duplicate"
                                log["status"] = "duplicate"
                            else:
                                db.execute("INSERT INTO hashes(content_hash,uid) VALUES(?,?)", (doc["content_hash"],doc["uid"]))
                        db.execute("UPDATE urls SET status='done',attempt=?,log_json=?,doc_json=? WHERE id=?",
                                   (log["attempt"],json.dumps(log,ensure_ascii=False),json.dumps(doc,ensure_ascii=False) if doc else None,int(row["id"])))
                        if result.get("host_blocked"):
                            db.execute("INSERT OR REPLACE INTO domain_blocks(domain,reason) VALUES(?,?)",(host,log["error"] or log["status"]))
                    session_counts[log["status"]] += 1
                    export_ready()
                    now = time.monotonic()
                    if now-last_report >= PROGRESS_SECONDS:
                        placeholders = ",".join("?" for _ in EXCLUDED_DOMAINS)
                        eligible = f"domain NOT IN ({placeholders})"
                        excluded = tuple(sorted(EXCLUDED_DOMAINS))
                        counts = {r["status"]:r["n"] for r in db.execute(
                            f"SELECT json_extract(log_json,'$.status') AS status,COUNT(*) AS n FROM urls WHERE status='done' AND {eligible} GROUP BY status",
                            excluded,
                        )}
                        total,done,pending = db.execute(
                            f"SELECT COUNT(*),SUM(status='done'),SUM(status!='done') FROM urls WHERE {eligible}", excluded
                        ).fetchone()
                        status_labels = [("blocked","bị chặn"),("too_short","quá ngắn"),("ok","Đạt"),("empty","rỗng"),("http_error","lỗi HTTP"),("timeout","quá giờ"),("lang_mismatch","sai ngôn ngữ"),("duplicate","trùng lặp")]
                        details = " | ".join(f"{label} ({code}) {counts.get(code,0):,}" for code,label in status_labels)
                        rate = sum(session_counts.values())/max(now-started,0.001)
                        eta = pending/rate/3600 if rate else 0
                        print(f"\rTiến độ A: {done or 0:,}/{total:,} | còn {pending or 0:,} | {details} | còn khoảng {eta:.1f} giờ",end="",flush=True)
                        last_report = now
                    if result.get("host_blocked"):
                        stop_hosts.add(host)
                        return
                except asyncio.CancelledError:
                    raise
        hosts = [r[0] for r in db.execute("SELECT DISTINCT domain FROM urls WHERE status='pending' ORDER BY domain")]
        tasks = [asyncio.create_task(crawl_domain(host)) for host in hosts]
        try:
            await asyncio.gather(*tasks)
        except BaseException:
            for task in tasks:
                task.cancel()
            await asyncio.gather(*tasks,return_exceptions=True)
            raise
        finally:
            export_ready(force=True)


def save_resume_bundle() -> None:
    db.commit()
    db.execute("PRAGMA wal_checkpoint(FULL)")
    db.close()
    temp_bundle = RESUME_BUNDLE.with_suffix(".zip.tmp")
    with zipfile.ZipFile(temp_bundle,"w",compression=zipfile.ZIP_DEFLATED) as archive:
        archive.write(STATE_PATH,arcname="state-a-current.sqlite3")
        for folder in (DOCS_DIR,LOGS_DIR):
            for path in sorted(folder.glob("*")):
                if path.is_file() and not path.name.endswith(".tmp"):
                    archive.write(path,arcname=str(path.relative_to(WORK_ROOT)))
    os.replace(temp_bundle,RESUME_BUNDLE)
    print(f"Đã lưu gói resume: {RESUME_BUNDLE}")

try:
    await crawl_all()
except KeyboardInterrupt:
    print("\nĐã nhận yêu cầu dừng; đang xuất kết quả và đóng checkpoint.")
finally:
    try:
        export_ready(force=True)
        show_summary()
        save_resume_bundle()
    except Exception:
        if db:
            db.close()
        raise